# 📘 Notebook 12: LSTMs and Text Generation

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module E: Neural Networks for Text · Notebook 1 of 3 in this module · (12 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a plain recurrent network **forgets**, and demonstrate it with a measurement
- Describe the **LSTM** cell: its cell state and its forget, input and output gates
- Compute one LSTM step **by hand** in NumPy and check it against PyTorch
- Build and train a **character-level language model** on a novel
- **Generate** text from the model and control it with **temperature**
- Explain what **stacked** and **bidirectional** LSTMs are and when each can be used

> **Prerequisites:** Notebooks 07 and 10 of this course. From *From Python to Deep Learning & Fine-Tuning*: Notebook 13 (PyTorch and the training loop) and Notebook 15 (the idea of a recurrent network and its hidden state). This notebook does not re-teach those. It starts where Notebook 15 stops.
>
> 🔭 **Why this notebook matters:** since Notebook 01 one row of our score table has read "no": none of our representations used the **order** of the words beyond a tiny window. Recurrent networks read a text one item at a time and carry a memory forward, so order is built into them. The LSTM is the recurrent network that made this work in practice, and for about a decade it was the engine of machine translation, speech recognition and predictive keyboards. It is also still a language model in the sense of Notebook 07, trained to predict what comes next and judged by perplexity.

> ℹ️ **Setup note.** PyTorch is already installed in Google Colab. Training in this notebook takes about a minute on an ordinary processor. No GPU is needed.

> 📦 **Libraries used in this notebook.**
>
> - **PyTorch** (installed under the name `torch`) is the deep-learning library most widely used in research, developed originally at Meta. It provides tensors, automatic differentiation and ready-made layers such as `nn.LSTM`. It is introduced in *From Python to Deep Learning & Fine-Tuning*, Notebook 13.
> - **NumPy** is used for the LSTM step we write by hand, and **NLTK** supplies the text of a novel.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment. On your own computer it may take a minute the first time.

In [ ]:
%pip install -q nltk numpy torch

In [ ]:
import nltk
nltk.download("gutenberg", quiet=True)

import math
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

## 1. The memory problem, measured

### Intuition first
The n-gram models of Notebook 07 had two limits. They could see only two or three words back, and they could not share knowledge between similar words. Word embeddings (Notebook 10) removed the second limit. A **recurrent neural network (RNN)** addresses the first. It reads a sequence one step at a time and keeps a **hidden state**, a vector that summarises everything read so far:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

In principle the hidden state can carry information from the first word to the last. In practice the plain RNN is forgetful. Think of the game in which a message is whispered along a line of people. Every person rewrites the message completely before passing it on, and after twenty people little of the original is left. In the RNN, every step rewrites the **whole** hidden state.

Let us measure it. We feed the same network two sequences that are identical except for their **first** input, and watch how different the two hidden states remain as the steps go by.

In [ ]:
rng = np.random.default_rng(0)
hidden_size = 16

W_h = rng.normal(0, 0.2, (hidden_size, hidden_size))
W_x = rng.normal(0, 0.5, (hidden_size, 1))

def run_rnn(first_input, steps):
    h = np.zeros(hidden_size)
    states = []
    for t in range(steps):
        x = first_input if t == 0 else 0.5            # only the first input differs
        h = np.tanh(W_x[:, 0] * x + W_h @ h)
        states.append(h.copy())
    return states

states_a = run_rnn(first_input=1.0, steps=40)
states_b = run_rnn(first_input=-1.0, steps=40)

print("Distance between the two hidden states")
for t in [0, 1, 2, 5, 10, 20, 39]:
    print(f"  after step {t + 1:>2}: {np.linalg.norm(states_a[t] - states_b[t]):.6f}")

The trace of the first input shrinks by a roughly constant factor at every step, and after a few dozen steps it has vanished. Nothing the network does later can depend on it.

The same shrinking happens to the gradients during training, flowing backwards. This is the **vanishing gradient problem**: the network receives no learning signal connecting an early cause to a late effect, so it cannot learn long-range dependencies such as the agreement between *keys* and *are* in *"The keys that I left on the table **are** missing"*.

## 2. The LSTM

### Intuition first
The **Long Short-Term Memory** network (Hochreiter and Schmidhuber, 1997) changes one thing. Alongside the hidden state it keeps a second vector, the **cell state**, which works like a notebook that the network carries along the sequence. The crucial difference is in how the notebook is updated. A plain RNN rewrites its whole memory at every step. An LSTM only **edits** its notebook: it may rub out a few lines, write a few new ones, and leave the rest exactly as it was.

Three **gates** control the editing. A gate is a vector of numbers between 0 and 1, one per entry of the notebook, acting as a dial: 0 means "block completely", 1 means "let through completely".

- The **forget gate** decides which entries to rub out.
- The **input gate** decides which new entries to write.
- The **output gate** decides which entries to read out for the task at hand.

The gates are not programmed. Each is a small layer whose weights are learned, so the network learns **what to remember, what to forget and when**.

### Formal definition
At step $t$, with input $x_t$, previous hidden state $h_{t-1}$ and previous cell state $c_{t-1}$:

$$\begin{aligned}
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) && \text{forget gate} \\
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) && \text{input gate} \\
g_t &= \tanh(W_g x_t + U_g h_{t-1} + b_g) && \text{candidate values} \\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) && \text{output gate} \\[4pt]
c_t &= f_t \odot c_{t-1} + i_t \odot g_t && \text{edit the notebook} \\
h_t &= o_t \odot \tanh(c_t) && \text{read from it}
\end{aligned}$$

Here $\sigma$ is the sigmoid function and $\odot$ multiplies element by element. The line that matters most is the fifth. The new cell state is the old one, **multiplied** by the forget gate, **plus** something new. When the forget gate is 1 and the input gate is 0, the cell state passes through a step completely unchanged. Information can travel across hundreds of steps along this path without being rewritten, and so can the gradient.

### One step by hand
The six equations in NumPy. The four sets of weights are stored stacked on top of each other, in the order used by PyTorch: input, forget, candidate, output.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def lstm_step(x, h, c, W, U, b):
    H = len(h)
    z = W @ x + U @ h + b                   # all four layers at once, stacked
    i = sigmoid(z[0:H])                     # input gate
    f = sigmoid(z[H:2 * H])                 # forget gate
    g = np.tanh(z[2 * H:3 * H])             # candidate values
    o = sigmoid(z[3 * H:4 * H])             # output gate
    c_new = f * c + i * g                   # edit the notebook
    h_new = o * np.tanh(c_new)              # read from it
    return h_new, c_new

Is this really what a professional library computes? We create a PyTorch LSTM cell, copy its random weights into our function, and compare the results on the same input.

In [ ]:
input_size, hidden_size = 3, 4
cell = nn.LSTMCell(input_size, hidden_size)

x = torch.randn(input_size)
h = torch.randn(hidden_size)
c = torch.randn(hidden_size)

with torch.no_grad():
    h_torch, c_torch = cell(x.unsqueeze(0), (h.unsqueeze(0), c.unsqueeze(0)))

W = cell.weight_ih.detach().numpy()
U = cell.weight_hh.detach().numpy()
b = (cell.bias_ih + cell.bias_hh).detach().numpy()
h_ours, c_ours = lstm_step(x.numpy(), h.numpy(), c.numpy(), W, U, b)

print("PyTorch h:", np.round(h_torch.numpy()[0], 4))
print("Ours    h:", np.round(h_ours, 4))
print("Largest difference:", float(np.abs(h_torch.numpy()[0] - h_ours).max()))

The two agree to within rounding error. There is no hidden machinery in an LSTM: nine lines of arithmetic, repeated at every step.

### The memory, measured again
To see the difference from Section 1, let us set the gates by hand to "keep everything, write nothing" and run the cell for many steps.

In [ ]:
H = 4
W = np.zeros((4 * H, 1))
U = np.zeros((4 * H, H))
b = np.zeros(4 * H)
b[0:H] = -10            # input gate: sigmoid(-10) is almost 0, so nothing new is written
b[H:2 * H] = 10         # forget gate: sigmoid(10) is almost 1, so everything is kept

h = np.zeros(H)
c = np.array([1.0, -2.0, 0.5, 3.0])        # something written in the notebook earlier
print("start:      ", c)
for step in range(1, 201):
    h, c = lstm_step(np.array([0.7]), h, c, W, U, b)
    if step in (1, 50, 200):
        print(f"step {step:>3}:   ", np.round(c, 3))

After 200 steps the contents are almost untouched. A trained LSTM does not keep its gates in this extreme position, of course. The point is that it **can** hold a value for as long as it finds useful, which a plain RNN cannot.

## 3. A character-level language model

Time to train one. Our task is the task of Notebook 07, predicting what comes next, with two changes. The model is an LSTM, and the units are **characters** and not words. A character model has a tiny vocabulary and must learn everything else from scratch: how to spell words, where spaces go, how sentences are punctuated.

### The data
We use *Alice in Wonderland*, lower-cased, with all runs of whitespace turned into single spaces. Each distinct character gets a number.

In [ ]:
from nltk.corpus import gutenberg

text = gutenberg.raw("carroll-alice.txt").lower()
text = re.sub(r"\s+", " ", text)

characters = sorted(set(text))
char_id = {character: i for i, character in enumerate(characters)}
vocabulary_size = len(characters)

print(f"{len(text):,} characters, {vocabulary_size} distinct")
print("".join(characters))

data = torch.tensor([char_id[character] for character in text])
cut = int(0.9 * len(data))
train_data, validation_data = data[:cut], data[cut:]
print(data[:20].tolist(), "=", repr(text[:20]))

The training examples are stretches of 64 characters. For each position, the target is simply the **next** character, so the target sequence is the input shifted by one.

In [ ]:
def get_batch(source, batch_size=64, length=64):
    starts = torch.randint(0, len(source) - length - 1, (batch_size,))
    inputs = torch.stack([source[s:s + length] for s in starts])
    targets = torch.stack([source[s + 1:s + length + 1] for s in starts])
    return inputs, targets

inputs, targets = get_batch(train_data, batch_size=1, length=24)
print("input: ", repr("".join(characters[i] for i in inputs[0])))
print("target:", repr("".join(characters[i] for i in targets[0])))

### The model
Three layers, each of which you have met before:

1. an **embedding** layer gives each character a vector (the embedding table of Notebook 10, learned as part of the model);
2. an **LSTM** reads the vectors in order and produces a hidden state at every position;
3. a **linear** layer turns each hidden state into one score per character of the vocabulary. A softmax over those scores is the predicted probability of the next character.

In [ ]:
class CharacterModel(nn.Module):
    def __init__(self, vocabulary_size, embedding_size=32, hidden_size=128, kind="lstm", layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size)
        recurrent_layer = nn.LSTM if kind == "lstm" else nn.GRU
        self.recurrent = recurrent_layer(embedding_size, hidden_size, num_layers=layers, batch_first=True)
        self.output = nn.Linear(hidden_size, vocabulary_size)

    def forward(self, inputs, state=None):
        vectors = self.embedding(inputs)                    # (batch, length, embedding_size)
        hidden, state = self.recurrent(vectors, state)      # (batch, length, hidden_size)
        return self.output(hidden), state                   # (batch, length, vocabulary_size)

model = CharacterModel(vocabulary_size)
scores, state = model(inputs)
print("Input shape: ", tuple(inputs.shape))
print("Output shape:", tuple(scores.shape))
print("Parameters:  ", f"{sum(p.numel() for p in model.parameters()):,}")

### Training
The loop is the standard one from the companion course: predict, measure the loss, back-propagate, update. The loss is **cross-entropy**, the average of $-\log P(\text{correct next character})$. That is exactly the quantity inside the perplexity formula of Notebook 07, so

$$\text{perplexity} = e^{\text{loss}}$$

Before any training the model knows nothing, and its perplexity should be close to the vocabulary size.

In [ ]:
def evaluate(model, batches=20):
    model.eval()
    generator_state = torch.random.get_rng_state()
    torch.manual_seed(123)                                   # always the same validation batches
    total = 0.0
    with torch.no_grad():
        for i in range(batches):
            inputs, targets = get_batch(validation_data)
            scores, state = model(inputs)
            total += F.cross_entropy(scores.reshape(-1, vocabulary_size), targets.reshape(-1)).item()
    torch.random.set_rng_state(generator_state)
    model.train()
    return total / batches

def train(model, steps, learning_rate=3e-3, report_every=100):
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for step in range(1, steps + 1):
        inputs, targets = get_batch(train_data)
        scores, state = model(inputs)
        loss = F.cross_entropy(scores.reshape(-1, vocabulary_size), targets.reshape(-1))
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)    # keep the gradients from exploding
        optimiser.step()
        if step % report_every == 0:
            validation_loss = evaluate(model)
            print(f"step {step:>4}  training loss {loss.item():.3f}  validation loss {validation_loss:.3f}  perplexity {math.exp(validation_loss):.1f}")

start_loss = evaluate(model)
print(f"Before training: loss {start_loss:.3f}, perplexity {math.exp(start_loss):.1f} (vocabulary size {vocabulary_size})\n")

train(model, steps=800)

The perplexity started near the vocabulary size and has fallen to a small number. On average the model is now about as uncertain as if it were choosing between a handful of characters at each step, having started with all of them equally likely. Remember that it was told nothing about English: not what a word is, not that `q` is followed by `u`.

## 4. Generating text

To write, we do what Notebook 07 did with n-grams: predict a distribution over the next character, **sample** from it, feed the sampled character back in, and repeat. The LSTM's state is carried along, so the model remembers what it has written.

### Temperature
One extra dial is useful. Before the softmax we divide the scores by a number $T$, the **temperature**:

$$P(\text{character}) = \text{softmax}(\text{scores} / T)$$

- With $T < 1$ the differences between scores are exaggerated. The model becomes cautious and keeps choosing its favourite continuations.
- With $T > 1$ the differences are flattened. The model takes more risks.
- $T = 1$ samples from the distribution exactly as the model learned it.

In [ ]:
def generate(model, start="alice ", length=300, temperature=0.8):
    model.eval()
    written = list(start)
    with torch.no_grad():
        inputs = torch.tensor([[char_id[character] for character in start]])
        scores, state = model(inputs)
        for i in range(length):
            probabilities = torch.softmax(scores[0, -1] / temperature, dim=0)
            next_id = torch.multinomial(probabilities, 1).item()
            written.append(characters[next_id])
            scores, state = model(torch.tensor([[next_id]]), state)
    model.train()
    return "".join(written)

torch.manual_seed(1)
for temperature in [0.3, 0.8, 1.5]:
    print(f"--- temperature {temperature}")
    print(generate(model, temperature=temperature, length=260))
    print()

Read the three samples.

- At **low temperature** the text is made of real, common words and safe phrases, and it tends to go round in circles.
- At **medium temperature** it is more varied. Most words are real, the punctuation and quotation marks are plausible, and there is a rough imitation of the rhythm of the book.
- At **high temperature** it becomes inventive to the point of nonsense, with many invented words.

None of it means anything, and that is worth being clear about. One minute of training on one short book gives a model that has learned spelling and local phrasing, and nothing of the plot. But notice what it has worked out from raw characters: words, spaces, common endings, the fact that a quotation once opened is usually closed. Larger versions of exactly this procedure, with more data and bigger networks, were the state of the art in language modelling until 2017.

> ⚠️ **Common pitfalls**
>
> - Forgetting to pass the **state** back in during generation. Without it the model sees one character at a time with no memory, and produces a jumble.
> - Always taking the single most probable character. The output quickly falls into a loop, repeating the same phrase. Sampling is what makes generated text varied. Notebook 15 studies this choice in depth.
> - Judging a language model by its samples alone. They are entertaining and misleading. Compare models by their **validation loss** or perplexity.

## 5. Stacked and bidirectional LSTMs

Two variations are so common that you will meet them in almost any description of an LSTM system.

**Stacked.** Several LSTM layers are placed on top of one another. The sequence of hidden states produced by the first layer becomes the input sequence of the second. As with deep networks for images, lower layers tend to capture local patterns and higher layers more abstract ones.

**Bidirectional.** Two LSTMs read the same sequence, one from left to right and one from right to left, and their hidden states are joined at each position. Each position then has information about **both** what comes before it and what comes after.

In [ ]:
vectors = torch.randn(1, 10, 32)             # one sequence of 10 steps, 32 numbers per step

plain = nn.LSTM(32, 128, batch_first=True)
stacked = nn.LSTM(32, 128, num_layers=2, batch_first=True)
both_ways = nn.LSTM(32, 128, bidirectional=True, batch_first=True)

for name, layer in [("plain", plain), ("stacked, 2 layers", stacked), ("bidirectional", both_ways)]:
    hidden, (h, c) = layer(vectors)
    parameters = sum(p.numel() for p in layer.parameters())
    print(f"{name:<18} output {tuple(hidden.shape)}   parameters {parameters:,}")

The bidirectional output has 256 numbers per position: 128 from each direction.

A bidirectional LSTM is the natural choice whenever the **whole text is available** before a decision is needed: tagging parts of speech or named entities (Notebook 06), or classifying a document (Notebook 09). To decide whether *book* is a noun or a verb, the words after it help as much as the words before.

It **cannot** be used to generate text, and the reason is worth a moment's thought. A generator must predict the next character without seeing it. A network that reads from the right has already seen it.

> 🧠 The **GRU** (gated recurrent unit) is a slightly simpler relative of the LSTM, with two gates and no separate cell state. It has fewer parameters and often performs about as well. Our `CharacterModel` accepts `kind="gru"`, and an exercise below compares the two.

## 6. Where we stand

| | N-gram model | Word embeddings | LSTM language model |
|---|---|---|---|
| Uses word order | a few words back | no | **yes** |
| Shares knowledge between similar items | no | yes | yes, through its embedding layer |
| Reach of its memory | 2 or 3 words | none | tens to hundreds of steps |
| Trained by | counting | prediction | prediction |
| Can be run in parallel over a sequence | yes | yes | **no** |

For the first time, order is fully part of the model. Two weaknesses remain. The memory of an LSTM, though far better than that of a plain RNN, still fades over long distances. And because each step needs the result of the step before, an LSTM cannot process the positions of a sequence in parallel, which makes it slow to train on very large amounts of text. Both weaknesses are removed by **attention** and the **Transformer**, covered in Notebooks 15 and 16 of *From Python to Deep Learning & Fine-Tuning*.

Before going there, the next notebook puts two LSTMs together to do something a single one cannot: turn one sequence into a **different** sequence.

---
## ✏️ Exercises

### Exercise 1 (One gate at a time, on paper)
Consider an LSTM with a single cell. At some step the gates have the values $f = 0.9$, $i = 0.5$ and $o = 0.5$, the candidate value is $g = 0.8$, and the previous cell state is $c_{t-1} = 1.0$. Compute the new cell state $c_t$ and the hidden state $h_t$ on paper, then check in Python. What would $c_t$ be if the forget gate were 0?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
f, i, o, g, c_previous = 0.9, 0.5, 0.5, 0.8, 1.0

c_new = f * c_previous + i * g           # 0.9 * 1.0 + 0.5 * 0.8 = 1.3
h_new = o * np.tanh(c_new)               # 0.5 * tanh(1.3) = 0.431
print(c_new, round(float(h_new), 3))

print("With forget gate 0:", 0.0 * c_previous + i * g)
```

*Most of the old memory (90%) is kept and half of the candidate is added. With the forget gate at 0 the old content is erased, and only the new contribution of 0.4 remains.*
</details>

### Exercise 2 (Count the parameters)
An LSTM layer with input size $D$ and hidden size $H$ has four sets of weights, each consisting of a matrix of size $H \times D$, a matrix of size $H \times H$ and, in PyTorch, two bias vectors of size $H$. Compute the number of parameters of `nn.LSTM(32, 128)` with this formula, and check it by counting the parameters of an actual layer. How many would a plain RNN of the same size have?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
D, H = 32, 128
print("Formula:", 4 * (H * D + H * H + 2 * H))
print("PyTorch:", sum(p.numel() for p in nn.LSTM(D, H).parameters()))
print("Plain RNN:", sum(p.numel() for p in nn.RNN(D, H).parameters()))
```

*An LSTM has exactly four times the parameters of a plain RNN of the same size: one set for each gate and one for the candidate values.*
</details>

### Exercise 3 (Temperature, measured)
"More inventive" can be measured. For each temperature in `[0.2, 0.5, 0.8, 1.0, 1.5, 2.0]`, generate 1,500 characters, split the result into words, and compute the share of words that are **real words of the book** (words that occur in `text`). Use `torch.manual_seed(0)` before each generation.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
book_words = set(re.findall(r"[a-z]+", text))

for temperature in [0.2, 0.5, 0.8, 1.0, 1.5, 2.0]:
    torch.manual_seed(0)
    sample = generate(model, length=1500, temperature=temperature)
    words = re.findall(r"[a-z]+", sample)
    real = sum(1 for word in words if word in book_words)
    print(f"temperature {temperature:<4} {real / len(words):>5.0%} real words   {len(set(words)):>4} different words")
```

*As the temperature rises, the share of real words falls and the number of different words rises. Low temperature gives correct and repetitive text, high temperature gives varied and incorrect text. Every system that generates text has to choose a point on this scale.*
</details>

### Exercise 4 (LSTM or GRU?)
Train two fresh models for 300 steps each, one with `kind="lstm"` and one with `kind="gru"`, starting from the same random seed. Compare their validation loss and their number of parameters.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for kind in ["lstm", "gru"]:
    torch.manual_seed(0)
    candidate = CharacterModel(vocabulary_size, kind=kind)
    train(candidate, steps=300, report_every=1000)             # no progress reports
    loss = evaluate(candidate)
    parameters = sum(p.numel() for p in candidate.parameters())
    print(f"{kind:<5} {parameters:>8,} parameters   validation loss {loss:.3f}   perplexity {math.exp(loss):.1f}")
```

*The GRU has about a quarter fewer parameters, and on a task of this size it reaches an even lower loss in the same number of steps. Neither is better in general: on other tasks and with longer training the order is often reversed. With a fixed budget of training time it is worth trying both.*
</details>

### Exercise 5 (A second layer)
Does a stacked model do better? Train a model with `layers=2` for 800 steps from seed 0 and compare its validation loss with that of the one-layer `model` trained above.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
torch.manual_seed(0)
deeper = CharacterModel(vocabulary_size, layers=2)
train(deeper, steps=800, report_every=400)

print(f"\nOne layer:  validation loss {evaluate(model):.3f}")
print(f"Two layers: validation loss {evaluate(deeper):.3f}")
```

*The second layer brings a modest improvement, at more than twice the cost in parameters and time. Whether that is worth it depends on the task. Extra capacity pays off most when there is more data to learn from. Always compare against the simpler model before keeping a more complex one.*
</details>

### Exercise 6 (Can an LSTM count? A little harder)
Notebook 04 showed that no finite automaton can check whether brackets are balanced, because that requires counting. Can an LSTM learn to? The cell below creates balanced strings, and unbalanced ones made by swapping two characters of a balanced string. Train a small LSTM classifier on strings of up to 16 characters. Then test it on strings of the same length, and on strings of 24 to 30 characters, **longer than any it has seen**.

In [ ]:
import random

def is_balanced(string):
    depth = 0
    for character in string:
        depth += 1 if character == "(" else -1
        if depth < 0:
            return False
    return depth == 0

def random_balanced(pairs):
    string, open_now, remaining = "", 0, pairs
    while remaining > 0 or open_now > 0:
        if remaining > 0 and (open_now == 0 or random.random() < 0.5):
            string += "("
            open_now += 1
            remaining -= 1
        else:
            string += ")"
            open_now -= 1
    return string

def random_example(min_pairs, max_pairs):
    string = random_balanced(random.randint(min_pairs, max_pairs))
    if random.random() < 0.5:
        return string, 1
    while True:                                   # swap two characters until the string is unbalanced
        letters = list(string)
        a, b = random.sample(range(len(letters)), 2)
        letters[a], letters[b] = letters[b], letters[a]
        if not is_balanced("".join(letters)):
            return "".join(letters), 0

random.seed(0)
print([random_example(2, 8) for i in range(4)])
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def make_batch(count, min_pairs, max_pairs):
    examples = [random_example(min_pairs, max_pairs) for i in range(count)]
    longest = max(len(string) for string, label in examples)
    # 1 for "(", 2 for ")", and 0 as padding placed in FRONT, so that the last step is always a real character
    rows = [[0] * (longest - len(string)) + [1 if ch == "(" else 2 for ch in string] for string, label in examples]
    return torch.tensor(rows), torch.tensor([float(label) for string, label in examples])

class BracketClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(3, 8, padding_idx=0)
        self.lstm = nn.LSTM(8, 16, batch_first=True)
        self.output = nn.Linear(16, 1)

    def forward(self, inputs):
        hidden, state = self.lstm(self.embedding(inputs))
        return self.output(hidden[:, -1]).squeeze(1)      # decide from the last hidden state

random.seed(0)
torch.manual_seed(0)
classifier = BracketClassifier()
optimiser = torch.optim.Adam(classifier.parameters(), lr=0.01)

for step in range(600):
    inputs, labels = make_batch(64, 1, 8)
    loss = F.binary_cross_entropy_with_logits(classifier(inputs), labels)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

def accuracy(min_pairs, max_pairs):
    inputs, labels = make_batch(2000, min_pairs, max_pairs)
    with torch.no_grad():
        predicted = (classifier(inputs) > 0).float()
    return (predicted == labels).float().mean().item()

print(f"Lengths seen in training (2 to 16):  {accuracy(1, 8):.1%}")
print(f"Longer strings (24 to 30):           {accuracy(12, 15):.1%}")
```

*On strings of the lengths it was trained on, the LSTM is nearly perfect: it has learned something that behaves like a counter, which no finite automaton of fixed size could be. On longer strings it is less accurate. It learned an approximate counter that works within the range it practised, and not the exact rule. This is typical of neural networks: excellent inside the range of their training data, and less reliable outside it.*
</details>

## 🔑 Key takeaways

- A plain RNN **rewrites** its whole memory at every step, so the trace of early inputs, and the gradient, vanish.
- An **LSTM** keeps a **cell state** that is only **edited**: $c_t = f_t \odot c_{t-1} + i_t \odot g_t$. The **forget**, **input** and **output** gates are learned.
- An LSTM step is a few lines of arithmetic, and our NumPy version matches PyTorch exactly.
- A **character-level language model** is embedding, LSTM and linear layer, trained to predict the next character. Its **perplexity** is $e^{\text{loss}}$, the same measure as in Notebook 07.
- Text is generated by **sampling** repeatedly. **Temperature** trades correctness against variety.
- **Stacked** LSTMs add depth. **Bidirectional** LSTMs see both sides of each position, and suit tagging and classification, never generation.
- LSTMs are sequential by nature: their memory still fades over long distances, and they cannot be trained in parallel over a sequence.

---
**Next:** *Notebook 13: Sequence-to-Sequence Models and Translation*, where one LSTM reads a sequence and a second one writes a different sequence in response.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*